# 00b — Prepare the official Symile cohort and available reference labels

Run after notebook 00 and before notebook 01, in the graph environment. This notebook is self-contained and uses NumPy to verify admission-array alignment when data_npy is available. It reads the official train/val/test CSVs; it does not run foundation models.

[Official dataset documentation](https://physionet.org/content/symile-mimic/1.0.0/) describes patient-disjoint splits, repeated test retrieval rows, and 14 CheXpert CXR label columns. We preserve the splits and deduplicate admission rows. Alternate study-ID columns and standard MIMIC paths are supported; missing label columns remain unknown and are listed in the audit. Pairing is not a finding label. CheXpert labels are report-derived reference annotations, not expert-certified ground truth.

This step produces `manifest.json`, `cxr_annotations.json`, `sensor_sources.json`, and an audit summary. **It does not yet produce embeddings, specialist candidates, or ECG reference annotations.** The latter are required before notebook 01 can create final exports. Do not flatten raw sensor tensors and call them learned embeddings.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'core.py').is_file(), 'Start in graph_reasoning/'
CONFIG = json.loads((PROJECT_ROOT / 'config.json').read_text())
SYMILE_ROOT = Path(CONFIG.get('symile_root', '/path/to/data/symile/physionet.org/files/symile-mimic/1.0.0')).expanduser().resolve()
PREPARED_DIR = PROJECT_ROOT / 'data' / 'prepared'
# Optional overrides if the CSVs are stored in a nonstandard layout:
CSV_PATHS = {'train': None, 'val': None, 'test': None}
# Optional independent ECG findings already mapped to the manifest IDs:
ECG_ANNOTATIONS_PATH = None

print('Symile directory:', SYMILE_ROOT)
print('Exists:', SYMILE_ROOT.is_dir())
if not SYMILE_ROOT.is_dir():
    raise FileNotFoundError('Run this notebook on Biowulf or set SYMILE_ROOT to an accessible dataset directory')
print('Top-level filenames only:')
for path in sorted(SYMILE_ROOT.iterdir())[:40]:
    print(path.name, '[directory]' if path.is_dir() else '[file]')


## CSV adapter
The following functions preserve identifiers as strings, resolve known study-ID aliases or standard MIMIC source paths, audit duplicate admissions and patient splits, and retain uncertain/missing labels as unknown. This follows Experiment 4’s CSV/admission-array alignment checks without depending on its external-reference partition. They do not print patient rows.

In [ ]:
"""Read documented Symile-MIMIC v1 CSVs; no model inference or generated labels."""
import csv
import hashlib
import json
import re
from pathlib import Path

LABELS = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
          'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
          'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
          'Pneumothorax', 'Support Devices']

def locate(root, name):
    direct = root / name
    if direct.is_file(): return direct
    matches = sorted(root.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {len(matches)}. Set CSV_PATHS explicitly.')
    return matches[0]

def label_value(value):
    value = value.strip()
    if value.lower() in ('', 'nan', 'none', '-1', '-1.0'): return None
    if value in ('0', '0.0'): return 0
    if value in ('1', '1.0'): return 1
    raise ValueError('Unexpected CheXpert label value; review the source encoding')

def clean(value):
    value = str(value or '').strip()
    return '' if value.lower() in ('nan', 'none', 'null') else value


def numeric_id(value):
    value = clean(value)
    if re.fullmatch(r'\d+\.0', value): value = value[:-2]
    return value


def sensor_identity(row, modality):
    path = clean(row.get(f'{modality}_path'))
    aliases = (('ecg_study_id', 'ecg_adm', 'ecg_file_name') if modality == 'ecg'
               else ('cxr_study_id', 'study_id'))
    values = []
    sources = []
    for name in aliases:
        value = clean(row.get(name))
        if value:
            if name == 'ecg_file_name':
                value = Path(value).name
                if value.endswith(('.hea', '.dat')): value = value.rsplit('.', 1)[0]
            values.append(numeric_id(value)); sources.append(name)
    # MIMIC paths identify studies as an s<digits> directory.
    match = re.search(r'(?:^|/)s(\d+)(?:/|$)', path)
    if match:
        values.append(match.group(1)); sources.append(f'{modality}_path')
    if modality == 'ecg' and not match:
        name = Path(path).name
        if name.endswith(('.hea', '.dat')): name = name.rsplit('.', 1)[0]
        if name.isdigit(): values.append(name); sources.append('ecg_path')
    if not values:
        raise ValueError(f'Cannot identify {modality} study. Expected aliases {aliases} '
                         'or a standard MIMIC path containing s<study_id>.')
    if len(set(values)) != 1:
        raise ValueError(f'Conflicting {modality} study identities from {sources}')
    return values[0], sources


def cxr_image_identity(row):
    values = [clean(row.get(k)) for k in ('cxr_dicom_id', 'cxr_24_72_hr') if clean(row.get(k))]
    path = clean(row.get('cxr_path'))
    if Path(path).suffix.lower() in ('.jpg', '.jpeg', '.png', '.dcm'):
        values.append(Path(path).stem)
    if not values: raise ValueError('Cannot identify CXR image from aliases or image path')
    if len(set(values)) != 1: raise ValueError('Conflicting CXR image identities')
    return values[0]


def prepare(csv_paths, tensor_root=None):
    manifest = []; cxr_targets = []; sensors = []; patients = {}; admissions = set(); stats = {}
    required = ['subject_id', 'hadm_id', 'ecg_path', 'cxr_path']
    fingerprints = {}
    for split in ('train', 'val', 'test'):
        path = Path(csv_paths[split])
        fingerprints[split] = {'path': str(path), 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
        unique = {}; raw_count = 0; ordered_hadm = []
        identity_sources = {'ecg':set(), 'cxr':set()}
        with path.open(newline='', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            header = reader.fieldnames or []
            missing = set(required) - set(header)
            if missing: raise ValueError(f'{split}: missing required columns {sorted(missing)}; available: {header}')
            available_labels = [k for k in LABELS if k in header]
            absent_labels = [k for k in LABELS if k not in header]
            if not available_labels:
                raise ValueError(f'{split}: no recognized CXR label columns; inspect the CSV header before mapping labels')
            for row in reader:
                source_index = raw_count
                raw_count += 1
                if any(not clean(row.get(k)) for k in required):
                    raise ValueError(f'{split}: empty identity or sensor-path field at CSV row {raw_count+1}')
                admission = numeric_id(row['hadm_id'])
                row = dict(row, hadm_id=admission, subject_id=numeric_id(row['subject_id']))
                ordered_hadm.append(admission)
                for modality in ('ecg', 'cxr'):
                    study, sources = sensor_identity(row, modality)
                    row[f'{modality}_study_id'] = study
                    identity_sources[modality].update(sources)
                row['cxr_dicom_id'] = cxr_image_identity(row)
                findings = {label:label_value(row.get(label) or '') for label in LABELS}
                identity = tuple(clean(row.get(k)) for k in required + ['ecg_study_id', 'cxr_study_id',
                    'cxr_dicom_id', 'ecg_time', 'cxr_StudyDateTime', 'admittime']) + tuple(findings.values())
                if admission in unique:
                    if unique[admission][0] != identity:
                        raise ValueError(f'{split}: inconsistent repeated admission; refusing arbitrary selection')
                    continue
                unique[admission] = (identity, row, source_index, findings)
        if not unique: raise ValueError(f'{split}: empty cohort')
        tensor_check = 'not checked: no tensor_root supplied'
        if tensor_root is not None:
            import numpy as np
            ids_path = Path(tensor_root) / split / f'hadm_id_{split}.npy'
            if not ids_path.is_file(): raise FileNotFoundError(f'Missing admission identity array: {ids_path}')
            ids = np.load(ids_path, mmap_mode='r', allow_pickle=False).reshape(-1)
            if [numeric_id(str(v)) for v in ids] != ordered_hadm:
                raise ValueError(f'{split}: CSV/admission array identity mismatch')
            tensor_check = 'CSV row order matches hadm_id array'
            fingerprints[split]['hadm_array_sha256'] = hashlib.sha256(ids_path.read_bytes()).hexdigest()
        for admission, (_, row, source_index, findings) in unique.items():
            patient = row['subject_id']
            if admission in admissions: raise ValueError('Admission appears in multiple splits')
            if patient in patients and patients[patient] != split: raise ValueError('Patient leakage across official splits')
            admissions.add(admission); patients[patient] = split
            for modality in ('ecg','cxr'):
                rid = f'{admission}_{modality}'
                manifest.append(dict(id=rid,patient_id=patient,admission_id=admission,split=split,modality=modality))
                sensors.append(dict(id=rid,split=split,admission_id=admission,
                    source_index=source_index, source_csv=str(path),
                    source_path=row[f'{modality}_path'],study_id=row[f'{modality}_study_id'],
                    dicom_id=row['cxr_dicom_id'] if modality=='cxr' else None,
                    acquisition_time=row.get('ecg_time' if modality=='ecg' else 'cxr_StudyDateTime'),
                    admission_time=row.get('admittime')))
            cxr_targets.append(dict(id=f'{admission}_cxr', findings=findings,
                target_source='Symile-MIMIC v1.0.0 supplied CheXpert labels',
                target_revision='1.0.0; absent columns/-1/missing=unknown; explicit 0/1 retained'))
        stats[split] = dict(csv_rows=raw_count,unique_admissions=len(unique),
            repeated_rows_removed=raw_count-len(unique), available_cxr_label_columns=available_labels,
            unavailable_cxr_label_columns=absent_labels,
            identity_sources={k:sorted(v) for k,v in identity_sources.items()},
            tensor_identity_check=tensor_check)
    return manifest,cxr_targets,sensors,dict(splits=stats,patients=len(patients),
        admissions=len(admissions),source_files=fingerprints,
        status='Manifest and available CXR targets only; embeddings, specialist candidates and ECG targets still required')


def save_unchanged_or_new(path, value):
    path = Path(path); payload = json.dumps(value,indent=2,allow_nan=False)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise FileExistsError(f'{path}: different existing data; select a fresh prepared directory')
        return
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('x',encoding='utf-8') as f: f.write(payload)

def merge_annotations(manifest,cxr_targets,ecg_targets):
    expected = {r['id'] for r in manifest if r['modality']=='ecg'}
    if not isinstance(ecg_targets,list) or any(not isinstance(t,dict) for t in ecg_targets):
        raise ValueError('ECG annotations must be a JSON array of objects')
    ids = [t.get('id') for t in ecg_targets]
    if len(ids)!=len(set(ids)) or set(ids)!=expected:
        raise ValueError('ECG annotation IDs must match the manifest exactly, with no duplicates')
    for target in ecg_targets:
        if not target.get('target_source') or not target.get('target_revision'):
            raise ValueError('Independent ECG target provenance required')
        if not isinstance(target.get('findings'),dict) or not target['findings']:
            raise ValueError('ECG findings must contain independent annotations, not placeholder objects')
        if any(v not in (0,1,None) for v in target['findings'].values()):
            raise ValueError('ECG labels must be 1/0/null')
    return cxr_targets + ecg_targets


In [ ]:
CSV_PATHS = {split: Path(path).expanduser().resolve() if path else locate(SYMILE_ROOT, f'{split}.csv')
             for split, path in CSV_PATHS.items()}
tensor_root = SYMILE_ROOT / 'data_npy'
manifest, cxr_annotations, sensor_sources, preparation_audit = prepare(
    CSV_PATHS, tensor_root=tensor_root if tensor_root.is_dir() else None)
# Check all destinations before writing, so a differing earlier run is not overwritten.
exports = {'manifest.json':manifest, 'cxr_annotations.json':cxr_annotations,
           'sensor_sources.json':sensor_sources, 'symile_preparation_audit.json':preparation_audit}
for filename,value in exports.items():
    path=PREPARED_DIR/filename
    if path.exists() and json.loads(path.read_text()) != value:
        raise FileExistsError(f'{filename}: differs from existing output. Choose a fresh PREPARED_DIR.')
for filename,value in exports.items():
    save_unchanged_or_new(PREPARED_DIR/filename,value)
print('Saved:', ', '.join(exports))
print('Split counts:', preparation_audit['splits'])
for split, detail in preparation_audit['splits'].items():
    print(split, 'unavailable labels:', detail['unavailable_cxr_label_columns'])
    print(split, 'tensor check:', detail['tensor_identity_check'])
print('Patients:',preparation_audit['patients'],'Admissions:',preparation_audit['admissions'])


## Optional merge with independent ECG reference labels

The documented Symile CSV supplies CXR labels but does not document ECG interpretation labels. Link independent ECG reports/annotations using the selected `ecg_study_id` saved in `sensor_sources.json`, then map findings to a reviewed ECG ontology. Candidate PULSE output must not become its own target. Set `ECG_ANNOTATIONS_PATH` in the configuration cell only after that file exists.

Use the same concept spelling in candidates and targets. For CXR, targets retain the official CheXpert names, including spaces. `No Finding` is a dataset label, not a rule for turning unknown findings into negatives.

In [ ]:
if ECG_ANNOTATIONS_PATH is None:
    print('ECG annotations not supplied: annotations.json was not created.')
else:
    ecg_path=Path(ECG_ANNOTATIONS_PATH).expanduser()
    ecg_annotations=json.loads(ecg_path.read_text())
    annotations=merge_annotations(manifest,cxr_annotations,ecg_annotations)
    save_unchanged_or_new(PREPARED_DIR/'annotations.json',annotations)
    print('Saved combined independent annotations.json')

source_paths={name:str(PREPARED_DIR/f'{name}.json')
              for name in ('manifest','embeddings','candidates','annotations')}
print('Use these paths in notebook 01 SOURCE_PATHS:')
print(json.dumps(source_paths,indent=2))
for name,path in source_paths.items():
    print(name, 'exists:',Path(path).is_file())
print('Continue to notebook 01 only after all four prepared files exist.')


## Remaining sensor preparation

- `sensor_sources.json` records the source ECG/CXR paths and study IDs. These paths refer to the parent MIMIC datasets; they are not guaranteed to exist under SYMILE_ROOT.
- The published `data_npy` tensors are preprocessed images/signals, **not embeddings**. Before using them, map each row through `hadm_id_<split>.npy`; do not assume JSON/CSV row order after deduplication. Keep native timing and preprocessing provenance.
- Run PULSE in your working Python 3.9 environment on appropriately rendered ECG images, and MedGemma in its own environment on CXR images. Preserve record IDs from the manifest.
- Run frozen sensor encoders to produce `embeddings.json`; save checkpoints and preprocessing identifiers. Export model candidates as `candidates.json` using DATA_CONTRACT.md.
- This notebook does not automatically load large tensors, render waveforms, invoke models, or modify config.json. Those steps require the actual installed model interfaces and sensor-file layout. Share filenames/header information, not patient records, if an adapter fails.
